# Lecture 2. Interpreter, dependencies and the lockfile

Everything taught tonight, in the order it was taught. Run it top to bottom.

The examples use your own system, the credit decision. Every number in them is
an example rather than a recommendation. Which threshold you use, which policy
rules you add, which of the two mistakes you would rather make, and what the
applicant is told are yours to decide and to defend. Nothing here answers any
of those for you.

In [ ]:
# beat: 1630 which-python
import sys

print("interpreter:", sys.executable)
print("version:    ", sys.version.split()[0])
print("first three places import looks:")
for entry in sys.path[:3]:
    print("   ", entry or "(the current folder)")

In [ ]:
# beat: 1643 stdlib-against-pypi
import json

print("json ships with Python:", json.__file__)

try:
    import pandas
except ModuleNotFoundError as missing:
    print("pandas does not:", missing)

In [ ]:
# beat: 1647 where-did-it-go
from pathlib import Path

venv = Path(sys.prefix)
print("this environment lives at:", venv)
site_packages = next(venv.glob("lib/python*/site-packages"), None)
print("packages land in:         ", site_packages)
print("and that folder is on sys.path:", str(site_packages) in sys.path)

In [ ]:
# beat: 1749 generated-decide
import inspect

from decision_rules_example.generated import decide_as_generated as generated

source = inspect.getsource(generated.decide)
print(f"the assistant wrote {len(source.splitlines())} lines")
print("\n".join(source.splitlines()[:16]))

In [ ]:
# beat: 1806 line-count
from decision_rules_example import rules

after = inspect.getsource(rules.decide)
print("generated:              ", len(source.splitlines()), "lines")
print("after the three buckets:", len(after.splitlines()), "lines")

In [ ]:
# beat: 1811 fail-fast
from decision_rules_example.rules import Application, MissingValue, decide

incomplete = Application(age=34, monthly_income=6200.0, debt_ratio=None, late_payments=0)
try:
    decide(incomplete, 0.05)
except MissingValue as stopped:
    print("stopped, rather than treating the applicant as debt free:", stopped)

## Exercise 1. Make the failing check pass

One character below is wrong. The rule is that a probability **at or above**
the threshold is refused, and the third assertion says so. Run the cell, read
the failure, then fix it.

The threshold value itself is an example. Yours is a decision you make and
justify, and this exercise is about the comparison rather than the number.

In [ ]:
REFUSE_AT_OR_ABOVE = 0.30


def risk_is_acceptable(probability_of_default: float) -> bool:
    return probability_of_default <= REFUSE_AT_OR_ABOVE  # TODO one character is wrong


assert risk_is_acceptable(0.29)
assert not risk_is_acceptable(0.31)
assert not risk_is_acceptable(0.30), "at the threshold means refused"
print("all three pass")

<details>

<summary>Show the solution</summary>

```python
def risk_is_acceptable(probability_of_default: float) -> bool:
    """At or above the threshold means refused, so the comparison is strict."""
    return probability_of_default < REFUSE_AT_OR_ABOVE
```

The applicant sitting exactly on the threshold is the one the rule was written
for, and the one a loose comparison silently approves. Boundaries are where
rules are wrong, which is why the tests you write tonight go there first.

</details>

## Exercise 2. Write the missing test

An applicant under the minimum age is refused whatever the model says. That is
a policy rule, and no test covers it. Write one.

`Application` and `decide` are already imported above.

In [ ]:
def test_the_age_rule_overrides_a_low_probability():
    ...  # TODO


test_the_age_rule_overrides_a_low_probability()
print("it passes")

<details>

<summary>Show the solution</summary>

```python
def test_the_age_rule_overrides_a_low_probability(application):
    under_age = Application(**{**vars(application), "age": MINIMUM_AGE - 1})
    assert not decide(under_age, SAFE).approved
```

A policy rule is one the model cannot outvote, so the test has to start from an
application that would otherwise be approved. A test starting from a risky
application would pass whether the rule existed or not.

</details>